# Pneumonia CNN — Colab runbook

The notebook used to produce the results reported in `README.md`, run on a Colab
T4 GPU.

It expects `Archive.zip` (the dataset) and a zip of this repository to sit in a
folder named `nn` in Google Drive. Requires a GPU runtime:
**Runtime → Change runtime type → T4 GPU**.

## 1. Confirm the GPU runtime

In [ ]:
import tensorflow as tf
gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow", tf.__version__)
print("GPU:", gpus if gpus else "NONE - select a GPU runtime")

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import glob
FOLDER = '/content/drive/MyDrive/nn'

print("\nZip files found in", FOLDER)
for f in glob.glob(f'{FOLDER}/*.zip'):
    print("  ", f)

## 3. Unpack the code and the dataset

The two archives are told apart by their contents rather than their filenames.

In [ ]:
import glob, os, shutil, zipfile

zips = glob.glob(f'{FOLDER}/*.zip')
assert zips, f"No zip files in {FOLDER}"

code_zip = data_zip = None
for z in zips:
    with zipfile.ZipFile(z) as zf:
        names = zf.namelist()[:2000]
    if any('problem-set-01' in n for n in names):
        code_zip = z
    elif any(n.lower().endswith(('.jpeg', '.jpg')) for n in names):
        data_zip = z

assert code_zip, "Repository zip not found"
assert data_zip, "Dataset zip not found"
print("code   :", code_zip)
print("dataset:", data_zip)

shutil.copy(data_zip, '/content/Archive.zip')
with zipfile.ZipFile(code_zip) as zf:
    zf.extractall('/content/code')

PROJECT = glob.glob('/content/code/**/problem-set-01-pneumonia-cnn', recursive=True)[0]
os.chdir(PROJECT)
print("\nworking directory:", os.getcwd())

## 4. Prepare the data

Extracts the archive, removes the macOS metadata (`__MACOSX`, `.DS_Store`) that would
otherwise be indexed as image data, and reports the class distribution.

In [ ]:
!python src/prepare_data.py --zip /content/Archive.zip --out data/

## 5. MobileNetV2 transfer learning

Two stages: frozen backbone, then fine-tuning of the top 40 layers at lr=1e-5.

In [ ]:
!python src/train.py --data-dir data --model transfer --epochs 15

## 6. Evaluate the transfer model

Thresholds are selected on the validation split and then applied to the test set.

In [ ]:
!python src/evaluate.py --data-dir data --model-path outputs/models/transfer.keras

## 7. Baseline CNN trained from scratch

Trained for comparison against the pretrained backbone.

In [ ]:
!python src/train.py --data-dir data --model baseline --epochs 25

In [ ]:
!python src/evaluate.py --data-dir data --model-path outputs/models/baseline.keras

## 8. Results

In [ ]:
from IPython.display import Image, display
import glob, json

for f in sorted(glob.glob('outputs/*.json')):
    print("=" * 60, "\n", f)
    print(json.dumps(json.load(open(f)), indent=2))

for f in sorted(glob.glob('outputs/*.png')):
    print(f)
    display(Image(f))

## 9. Export the figures and metrics

Model weights are excluded; they are large and are not committed to the repository.

In [ ]:
!zip -r /content/outputs.zip outputs -x "outputs/models/*"
from google.colab import files
files.download('/content/outputs.zip')